In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text('catalog','fmcg','Catalog')
dbutils.widgets.text('data_source','orders','Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')



In [0]:
df = (
    spark.read.format('csv')
    .option('header','true')
    .option('inferSchema','true')
    .load('/Volumes/fmcg/bronze/source_fmcg/orders/landing')
    .withColumn('read_timestamp',F.current_timestamp())
    .select('*',"_metadata.file_name",'_metadata.file_size')
)
df.count()

In [0]:
df.write.format('delta')\
    .option('delta.enableChangeDataFeed','true')\
    .mode('append')\
    .saveAsTable('fmcg.bronze.orders')


In [0]:
df.display()

In [0]:
landing = '/Volumes/fmcg/bronze/source_fmcg/orders/landing/'
processed = '/Volumes/fmcg/bronze/source_fmcg/orders/processed/'
files = dbutils.fs.ls(landing)
for file_info in files: dbutils.fs.mv( file_info.path, f"{processed}/{file_info.name}", True )

In [0]:
df_orders =  spark.sql("select * from fmcg.bronze.orders.landing")
df_orders.display()

In [0]:
df_orders = df_orders.filter(F.col('order_qty').isNotNull())

In [0]:
df_orders = df_orders.withColumn( "customer_id", F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id")) .otherwise("999999") .cast("string") )

df_orders = df_orders.withColumn( "order_placement_date", F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "") )

#Pare the order_placement_Date using multiple possible formats

df_orders = df_orders.withColumn( "order_placement_date", F.coalesce( F.try_to_date("order_placement_date", "yyyy/MM/dd"), F.try_to_date("order_placement_date", "dd-MM-yyyy"), F.try_to_date("order_placement_date", "dd/MM/yyyy"), F.try_to_date("order_placement_date", "MMMM dd, yyyy"), ) )

#drop duplciates

df_orders = df_orders.dropDuplicates(['order_id','order_placement_date','customer_id','product_id','order_qty'])

df_orders = df_orders.withColumn('product_id',F.col('product_id').cast('string'))

df_orders.display()

In [0]:
df_orders.agg(
    F.min('order_placement_date').alias('min_date'),
    F.max('order_placement_date').alias('max_date')
).display()

In [0]:
df_products = spark.table('fmcg.silver.products')
df_products.display()

In [0]:
df_joined =df_orders.join(df_products,on = 'product_id',how = 'inner').select(df_orders['*'],df_products['product_code'])
df_joined.display()

In [0]:
if not (
    spark.catalog.tableExists('fmcg.silver.orders')
):
    print('creating new table')
    df_joined.write.format('delta').option('mergeSchema','true').mode('overwrite').saveAsTable('fmcg.silver.orders')
else:
    silver_delta = DeltaTable.forName(spark,'fmcg.silver.orders')
    silver_delta.alias('source')\
        .merge(df_joined.alias('joined'),'source.order_id = joined.order_id AND source.product_id = joined.product_id').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()



Gold

In [0]:
df_gold = spark.sql('SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM fmcg.silver.orders')
df_gold.display()

In [0]:
if not(spark.catalog.tableExists('fmcg.gold.sb_fact_orders')):
    print('creating table')
    df_gold.write.format('delta').option('deltaenableSchema','true').mode('overwrite').saveAsTable('fmcg.gold.sb_fact_orders')
else:
    gold_delta = DeltaTable.forName(spark,'fmcg.gold.sb_fact_orders')
    gold_delta.alias('source').merge(df_gold.alias('gold'),'source.order_id = gold.order_id').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

Merge with parent company


In [0]:
df_child = spark.sql('select date,product_code,customer_code,sold_quantity from fmcg.gold.sb_fact_orders')
df_child.display()

In [0]:
df_child.count()

In [0]:
df_monthly = (
    df_child.withColumn(
        'month_start',F.trunc('date','MM')
    )

.groupBy('month_start','product_code','customer_code')
.agg(
    F.sum('sold_quantity').alias('sold_quantity')
)
.withColumnRenamed('month_start','date')
)

In [0]:
gold_parent_delta = DeltaTable.forName(spark, "fmcg.gold.fact_orders") 

gold_parent_delta.alias("parent_gold").merge(df_monthly.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
files = dbutils.fs.ls(/Volumes/fmcg/bronze/source_fmcg/orders/landing/) for file_info in files: dbutils.fs.mv( file_info.path, f"{/Volumes/fmcg/bronze/source_fmcg/orders/processed/}/{file_info.name}", True )